# **01 - Descarga y Exploración de Datos Externos**

## Objetivo

El dataset original de partidos (`train.csv`) contiene información básica de cada encuentro: equipos, fecha, resultado y goles. Sin embargo, para construir un modelo predictivo competitivo, necesitamos **enriquecer estos datos** con variables externas que aporten contexto sobre la calidad y recursos de cada equipo.

En este cuaderno:
1. Explicamos la fuente de datos externa elegida.
2. Construimos y justificamos cada nueva variable.
3. Integramos las variables al dataset y guardamos el resultado en `data/interim/`.

## **Fuente de datos externa**

Los datos externos se han obtenido de un **[dataset de kaggle](https://www.kaggle.com/datasets/xfkzujqjvx97n/football-datasets/data?select=player_performances )**. El dataset combina datos de **[Transfermarkt](https://www.transfermarkt.es/)**, la mayor base de datos pública sobre fútbol profesional.

Los archivos descargados se encuentran en `data/raw/external/` e incluyen:

| Archivo | Descripción | Tamaño aprox. |
|---|---|---|
| `player_market_value.csv` | Valoración de mercado histórica de cada jugador | ~24 MB |
| `player_performances.csv` | Rendimiento de jugadores por temporada y equipo | ~158 MB |
| `player_profiles.csv` | Perfiles de jugadores (edad, nacionalidad, posición) | ~26 MB |
| `player_injuries.csv` | Historial de lesiones | ~8 MB |
| `team_details.csv` | Información general de equipos | ~0.6 MB |
| `team_competitions_seasons.csv` | Participación de equipos en competiciones | ~17 MB |
| `player_teammates_played_with.csv` | Relaciones entre compañeros de equipo | ~48 MB |
| `transfer_history.csv` | Historial de traspasos y fichajes de jugadores | ~48 MB |


### Reto principal: nombres de equipos

Los nombres de equipos en Transfermarkt son distintos a los del `train.csv`. Por ejemplo:

| `train.csv` | Transfermarkt |
|---|---|
| `Ath Bilbao` | `Athletic Bilbao` |
| `Betis` | `Real Betis Balompié` |
| `Man City` | `Manchester City` |
| `Wolves` | `Wolverhampton Wanderers` |

Para resolver esto, se ha construido un **diccionario de traducción completo** (`MAPEO_EQUIPOS` en `src/data_loader.py`) que cubre los 87 equipos presentes en el dataset.

## Setup inicial

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd

root_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_dir not in sys.path:
    sys.path.append(root_dir)

RUTA_TRAIN = '../data/raw/train.csv'
RUTA_VALUATIONS = '../data/raw/external/player_market_value.csv'
RUTA_PERFORMANCES = '../data/raw/external/player_performances.csv'
RUTA_TRANSFERS = '../data/raw/external/transfer_history.csv'
RUTA_SALIDA = '../data/interim/train_enriquecido.csv'

In [ ]:
df_partidos = pd.read_csv(RUTA_TRAIN)
df_partidos.head()

,id,division,anio_fin_temporada,fecha,local,visitante,goles_local,goles_visitante,resultado
0,10341_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Espanol,Almeria,3,3,X_empate
1,10334_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Osasuna,Girona,2,1,1_local
2,10335_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Real Madrid,Ath Bilbao,1,1,X_empate
3,10336_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Sociedad,Sevilla,2,1,1_local
4,10337_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Villarreal,Ath Madrid,2,2,X_empate


## **Variable 1: Valor de la plantilla al inicio de la temporada**

### ¿Por qué es interesante?

El **valor de mercado de una plantilla** es uno de los mejores indicadores de la calidad general de un equipo. Un equipo con jugadores más valiosos suele tener mayor probabilidad de ganar, ya que el valor de mercado refleja:
- El **talento individual** de los jugadores.
- La **inversión del club** en fichajes.


### Descripción de la variable

Para cada equipo y temporada, calculamos la **suma de las valoraciones de mercado de todos los jugadores de su plantilla** al inicio de la temporada.

Se generan dos columnas:
- `valor_plantilla_local`: Valor total de la plantilla del equipo local.
- `valor_plantilla_visitante`: Valor total de la plantilla del equipo visitante.

### Decisiones de diseño para evitar data leakage

En un modelo predictivo, es crucial no usar información que no estaría disponible en el momento de la predicción. Por ello, hemos decidido **Establecer como inicio de temporada el 1 de agosto**, ya que los partidos de La Liga y la Premier comienzan alrededor de principios de este mes, y aunque afectara a pocas jornadas, no queríamos incluir en el coste de la plantilla el valor de jugadores no presentes en los primeros partidos de la temporada. Aún cerrándose el mercado de fichajes el 1 de septiembre, los fichajes dedesde el 1 de julio al 1 de agosto suelen ser lo suficientemente significativos como para que haya cambios notables en el valor de la plantilla del equipo respecto a la temporada anterior.

### **Cálculo de la variable**

In [3]:
from src.data_loader import preparar_valor_plantilla_inicio_temporada

df_valor_plantilla = preparar_valor_plantilla_inicio_temporada(
    RUTA_VALUATIONS, RUTA_PERFORMANCES, RUTA_TRANSFERS
)
df_valor_plantilla.head(10)

,equipo,season_join_key,valor_plantilla_inicio
0,1. FC Eintracht Bamberg (- 2010),2009,350000.0
1,1.FC Eschborn (1930 - 2016),2009,0.0
2,1.FC Gera 03 (- 2012),2009,250000.0
3,1.FC Heidenheim 1846,2009,175000.0
4,1.FC Kaiserslautern,2009,14250000.0
5,1.FC Kaiserslautern II,2009,1000000.0
6,1.FC Kleve,2009,100000.0
7,1.FC Köln,2009,34350000.0
8,1.FC Köln II,2009,600000.0
9,1.FC Magdeburg,2009,225000.0


### **Integración con el dataset de partidos**

Hacemos **dos merges**: uno para el equipo local y otro para el visitante. La clave de unión es el nombre del equipo + el año de fin de temporada.

In [ ]:
df_enriquecido = pd.merge(
    df_partidos,
    df_valor_plantilla,
    left_on=['local', 'anio_fin_temporada'],
    right_on=['equipo', 'season_join_key'],
    how='left'
)
df_enriquecido = df_enriquecido.rename(columns={'valor_plantilla_inicio': 'valor_plantilla_local'})
df_enriquecido = df_enriquecido.drop(columns=['equipo', 'season_join_key'])

df_enriquecido = pd.merge(
    df_enriquecido,
    df_valor_plantilla,
    left_on=['visitante', 'anio_fin_temporada'],
    right_on=['equipo', 'season_join_key'],
    how='left'
)
df_enriquecido = df_enriquecido.rename(columns={'valor_plantilla_inicio': 'valor_plantilla_visitante'})
df_enriquecido = df_enriquecido.drop(columns=['equipo', 'season_join_key'])

df_enriquecido.head()

,id,division,anio_fin_temporada,fecha,local,visitante,goles_local,goles_visitante,resultado,valor_plantilla_local,valor_plantilla_visitante
0,10341_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Espanol,Almeria,3,3,X_empate,125050000.0,74950000.0
1,10334_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Osasuna,Girona,2,1,1_local,115700000.0,39550000.0
2,10335_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Real Madrid,Ath Bilbao,1,1,X_empate,866000000.0,216400000.0
3,10336_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Sociedad,Sevilla,2,1,1_local,368600000.0,264700000.0
4,10337_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Villarreal,Ath Madrid,2,2,X_empate,313200000.0,676500000.0


### **Validación de la variable**
Para mostrar que la variable funciona correctamente y tiene datos coherentes, probamos a listar los 10 equipos con mayor valor de plantilla para la temporada 2022/2023 

In [5]:
# Top 10 equipos con mayor valor de plantilla (temporada 2022/23)
top_equipos = (
    df_enriquecido[df_enriquecido['anio_fin_temporada'] == 2023]
    [['local', 'valor_plantilla_local']]
    .drop_duplicates('local')
    .sort_values('valor_plantilla_local', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top_equipos['valor_plantilla_local'] = (top_equipos['valor_plantilla_local'] / 1e6).round(0).astype(int).astype(str) + 'M €'
top_equipos.columns = ['Equipo', 'Valor plantilla']
top_equipos

,Equipo,Valor plantilla
0,Man City,1025M €
1,Real Madrid,866M €
2,Liverpool,865M €
3,Chelsea,841M €
4,Barcelona,831M €
5,Tottenham,757M €
6,Man United,730M €
7,Arsenal,717M €
8,Ath Madrid,676M €
9,Aston Villa,511M €


## **Variables 2 y 3: Altura media y edad media de la plantilla**

### ¿Por qué son interesantes?
- **Altura media**: En ciencia de datos nunca se sabe el impacto real que puede tener una variable aparentemente intrascendente hasta que se prueba. La altura promedio puede ser clave para determinar qué equipo tiene ventaja en **jugadas a balón parado** (córners, faltas), lo cual es una fuente importante de goles.
- **Edad media**: Nos permite capturar la **experiencia vs juventud** del equipo. Equipos muy jóvenes pueden ser más dinámicos pero menos constantes, mientras que equipos veteranos pueden tener más experiencia pero sufrir más el desgaste físico a lo largo de la temporada.

### Decisiones para evitar Data Leakage
Al igual que con el valor de la plantilla, la altura es estática y la edad se calcula a fecha **1 de agosto** del inicio de la temporada. La plantilla de cada equipo se determina usando el último traspaso conocido antes de esa misma fecha, garantizando que no se utiliza información futura.

In [ ]:
from src.data_loader import preparar_altura_edad_plantilla

RUTA_PROFILES = '../data/raw/external/player_profiles.csv'

df_altura_edad = preparar_altura_edad_plantilla(
    RUTA_PROFILES, RUTA_TRANSFERS, RUTA_PERFORMANCES
)

df_enriquecido = pd.merge(
    df_enriquecido,
    df_altura_edad,
    left_on=['local', 'anio_fin_temporada'],
    right_on=['equipo', 'season_join_key'],
    how='left'
)
df_enriquecido = df_enriquecido.rename(columns={
    'altura_media_plantilla': 'altura_media_local',
    'edad_media_plantilla': 'edad_media_local'
}).drop(columns=['equipo', 'season_join_key'])

df_enriquecido = pd.merge(
    df_enriquecido,
    df_altura_edad,
    left_on=['visitante', 'anio_fin_temporada'],
    right_on=['equipo', 'season_join_key'],
    how='left'
)
df_enriquecido = df_enriquecido.rename(columns={
    'altura_media_plantilla': 'altura_media_visitante',
    'edad_media_plantilla': 'edad_media_visitante'
}).drop(columns=['equipo', 'season_join_key'])

df_enriquecido.head()

,id,division,anio_fin_temporada,fecha,local,visitante,goles_local,goles_visitante,resultado,valor_plantilla_local,valor_plantilla_visitante,altura_media_local,edad_media_local,altura_media_visitante,edad_media_visitante
0,10341_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Espanol,Almeria,3,3,X_empate,125050000.0,74950000.0,179.7,26.83,181.4,25.23
1,10334_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Osasuna,Girona,2,1,1_local,115700000.0,39550000.0,181.1,27.66,181.6,26.14
2,10335_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Real Madrid,Ath Bilbao,1,1,X_empate,866000000.0,216400000.0,181.3,27.21,181.9,26.71
3,10336_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Sociedad,Sevilla,2,1,1_local,368600000.0,264700000.0,181.5,26.02,180.7,28.52
4,10337_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Villarreal,Ath Madrid,2,2,X_empate,313200000.0,676500000.0,181.1,28.10,181.6,27.44


## **Variables 4 y 5: Número y coste de jugadores lesionados**

### ¿Por qué son interesantes?
Las lesiones merman la capacidad competitiva de un equipo. Además, no es lo mismo perder a un jugador de rotación que a la estrella del equipo. Por eso, calculamos:
- **Número de lesionados**: Cuántas bajas tiene el equipo para ese partido.
- **Coste de lesionados**: Suma del valor de mercado de los jugadores que no pueden jugar. Penaliza más las ausencias de jugadores clave.

### Decisiones para evitar Data Leakage
Solo se consideran lesiones cuya fecha de inicio (`from_date`) sea **anterior o igual a la fecha del partido** para no tomar por error jugadores lesionados que no lo estén o jugadores que se hayan lesionado en ese mismo partido

In [7]:
from src.data_loader import preparar_lesiones_por_partido

RUTA_INJURIES = '../data/raw/external/player_injuries.csv'

df_lesiones = preparar_lesiones_por_partido(
    RUTA_INJURIES, RUTA_TRANSFERS, RUTA_VALUATIONS, 
    RUTA_PERFORMANCES, df_partidos
)

df_enriquecido = pd.merge(df_enriquecido, df_lesiones, on='id', how='left')

df_enriquecido[['num_lesionados_local', 'num_lesionados_visitante']] = df_enriquecido[['num_lesionados_local', 'num_lesionados_visitante']].fillna(0).astype(int)
df_enriquecido[['coste_lesionados_local', 'coste_lesionados_visitante']] = df_enriquecido[['coste_lesionados_local', 'coste_lesionados_visitante']].fillna(0)

df_enriquecido.head()

,id,division,anio_fin_temporada,fecha,local,visitante,goles_local,goles_visitante,resultado,valor_plantilla_local,valor_plantilla_visitante,altura_media_local,edad_media_local,altura_media_visitante,edad_media_visitante,num_lesionados_local,num_lesionados_visitante,coste_lesionados_local,coste_lesionados_visitante
0,10341_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Espanol,Almeria,3,3,X_empate,125050000.0,74950000.0,179.7,26.83,181.4,25.23,0,1,0.0,1800000.0
1,10334_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Osasuna,Girona,2,1,1_local,115700000.0,39550000.0,181.1,27.66,181.6,26.14,2,4,7000000.0,12100000.0
2,10335_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Real Madrid,Ath Bilbao,1,1,X_empate,866000000.0,216400000.0,181.3,27.21,181.9,26.71,0,4,0.0,38000000.0
3,10336_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Sociedad,Sevilla,2,1,1_local,368600000.0,264700000.0,181.5,26.02,180.7,28.52,1,0,18000000.0,0.0
4,10337_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Villarreal,Ath Madrid,2,2,X_empate,313200000.0,676500000.0,181.1,28.10,181.6,27.44,2,5,30000000.0,151000000.0


## **Guardado del dataset enriquecido**

Guardamos el dataset con las nuevas variables en `data/interim/train_enriquecido.csv`. Este archivo será el punto de partida para los siguientes cuadernos de análisis y modelado.

**Nuevas columnas añadidas:**
- `valor_plantilla_local` / `valor_plantilla_visitante`: Valor de la plantilla al inicio de temporada.
- `altura_media_local` / `altura_media_visitante`: Altura media de la plantilla al inicio de temporada.
- `edad_media_local` / `edad_media_visitante`: Edad media de la plantilla al inicio de temporada.
- `num_lesionados_local` / `num_lesionados_visitante`: Número de jugadores lesionados para el partido.
- `coste_lesionados_local` / `coste_lesionados_visitante`: Suma del valor de mercado de los jugadores lesionados.

In [ ]:
df_enriquecido.to_csv(RUTA_SALIDA, index=False)

print(f'Dataset guardado en: {RUTA_SALIDA}')


Dataset guardado en: ../data/interim/train_enriquecido.csv
